In [17]:
# make_tile_map_pro.py
"""
Generates a professional, interactive CONUS tile-index map:

  - Esri (ArcGIS) basemap, with a Street / Imagery layer toggle
  - Click-to-toggle tile selection (on/off, same as your existing selectable map)
  - Bounding-box selection, two ways:
        * drag a rectangle on the map (Leaflet.Draw)
        * type min/max lat/lon into the panel
    Either method selects every tile that intersects the box.
  - Coordinate lookup: type a lat/lon, the tile containing that point is
    selected and the map flies to it
  - Find-by-ID: type a tile_id, jump to and select that tile
  - Selection panel: live count, scrollable list, "Copy IDs" (comma-separated),
    "Clear"

Run with:   python make_tile_map_pro.py
Requires:   geopandas, folium, shapely  (pip install geopandas folium shapely)

All spatial selection logic (bbox intersect, point-in-polygon) runs
client-side in the browser via turf.js — no server, no re-execution needed
after the HTML is generated. Open the output file directly in any browser.
"""

import json
import geopandas as gpd
import folium
from folium import Element

# ----------------------------------------------------------------------
# CONFIG — edit these paths for your machine
# ----------------------------------------------------------------------
GPKG_PATH = r"C:/Users/mgvhy/OneDrive - University of Missouri/scientific_data/code_index/conus_tile_index.gpkg"
OUT_HTML = r"C:/Users/mgvhy/OneDrive - University of Missouri/scientific_data/code_index/conus_tile_map.html"
TILE_ID_FIELD = "tile_id"  # name of the tile-ID column in your gpkg

ESRI_STREET_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
ESRI_IMAGERY_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
ESRI_ATTR = "Tiles &copy; Esri"

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------
gdf = gpd.read_file(GPKG_PATH)

if TILE_ID_FIELD not in gdf.columns:
    raise ValueError(
        f"Column '{TILE_ID_FIELD}' not found in {GPKG_PATH}. "
        f"Available columns: {list(gdf.columns)}"
    )

# turf.js / Leaflet expect WGS84 lat/lon
if gdf.crs is None:
    raise ValueError("Input layer has no CRS set — cannot reproject to EPSG:4326.")
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

minx, miny, maxx, maxy = gdf.total_bounds
center_lat, center_lon = (miny + maxy) / 2, (minx + maxx) / 2
n_tiles = len(gdf)

geojson_data = json.loads(gdf.to_json())

# ----------------------------------------------------------------------
# BASE MAP
# ----------------------------------------------------------------------
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=4,
    tiles=None,
    control_scale=True,
    zoom_control=True,
)

folium.TileLayer(
    tiles=ESRI_STREET_URL,
    attr=ESRI_ATTR,
    name="Esri World Street Map",
    overlay=False,
    control=True,
).add_to(m)

folium.TileLayer(
    tiles=ESRI_IMAGERY_URL,
    attr=ESRI_ATTR,
    name="Esri World Imagery",
    overlay=False,
    control=True,
).add_to(m)

# ----------------------------------------------------------------------
# TILE LAYER
# ----------------------------------------------------------------------
geojson_layer = folium.GeoJson(
    geojson_data,
    name="Tile index",
    style_function=lambda x: {
        "fillColor": "#3186cc",
        "color": "#1a1a1a",
        "weight": 0.6,
        "fillOpacity": 0.22,
    },
    tooltip=folium.GeoJsonTooltip(fields=[TILE_ID_FIELD], aliases=["Tile ID:"]),
)
geojson_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# ----------------------------------------------------------------------
# EXTERNAL LIBS (Leaflet.Draw for drag-a-box, turf.js for spatial ops)
# ----------------------------------------------------------------------
m.get_root().header.add_child(Element(
    '<link rel="stylesheet" '
    'href="https://cdn.jsdelivr.net/npm/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>'
))
# defer is essential here: folium renders its own core dependencies
# (leaflet.js, jquery, etc.) as part of its default_js block, which is
# emitted AFTER anything added via header.add_child(). Without defer,
# leaflet-draw.js would run first and crash with "L is not defined"
# because Leaflet core hasn't loaded yet. defer guarantees this script
# runs after every regular (non-deferred) script has finished, and still
# before DOMContentLoaded — regardless of source order.
m.get_root().header.add_child(Element(
    '<script defer src="https://cdn.jsdelivr.net/npm/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>'
))
m.get_root().header.add_child(Element(
    '<script defer src="https://cdn.jsdelivr.net/npm/@turf/turf@6/turf.min.js"></script>'
))

# ----------------------------------------------------------------------
# PANEL UI (CSS + HTML)
# ----------------------------------------------------------------------
panel_css = """
<style>
#tile-panel {
    position: fixed; top: 10px; right: 10px; z-index: 9999;
    background: #ffffff; border: 1px solid #d0d0d0; border-radius: 8px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.15);
    width: 300px; font-family: "Segoe UI", Arial, sans-serif; font-size: 13px;
    color: #222;
}
#tile-panel h3 {
    margin: 0; padding: 10px 12px; background: #1f3a5f; color: #fff;
    font-size: 14px; border-radius: 8px 8px 0 0;
}
#tile-panel .section {
    padding: 10px 12px; border-top: 1px solid #eee;
}
#tile-panel label { display: block; font-weight: 600; margin-bottom: 4px; color: #444; }
#tile-panel input[type=text], #tile-panel input[type=number] {
    width: 100%; box-sizing: border-box; padding: 4px 6px; margin-bottom: 6px;
    border: 1px solid #ccc; border-radius: 4px; font-size: 12px;
}
#tile-panel .coord-row { display: flex; gap: 6px; }
#tile-panel .coord-row input { width: 50%; }
#tile-panel button {
    width: 100%; padding: 6px; margin-top: 2px; border: none; border-radius: 4px;
    background: #1f3a5f; color: #fff; font-size: 12px; cursor: pointer;
}
#tile-panel button.secondary { background: #6c757d; }
#tile-panel button.active { background: #c0392b; }
#tile-panel button:hover { opacity: 0.9; }
#sel-count { font-weight: 700; color: #1f3a5f; }
#sel-list {
    max-height: 130px; overflow-y: auto; border: 1px solid #eee; border-radius: 4px;
    padding: 6px; background: #fafafa; font-family: monospace; font-size: 11px;
    margin-bottom: 6px; word-break: break-all;
}
#tile-panel .meta { color: #777; font-size: 11px; margin-top: 4px; }
#tile-panel .status { font-size: 11px; color: #1f3a5f; min-height: 14px; margin-top: 4px; }

/* Leaflet gives every tile polygon its own "pointer" cursor (leaflet-interactive),
   which sits on top of the map container and wins the cursor regardless of
   CSS specificity tricks. The reliable fix: while draw mode is active, make
   the tiles stop intercepting the mouse (pointer-events: none) so the
   cursor falls through to the container's crosshair underneath. This also
   stops accidental tile-clicks while you're mid-drag. */
.leaflet-container.drawing-active {
    cursor: crosshair !important;
}
.leaflet-container.drawing-active .leaflet-interactive {
    cursor: crosshair !important;
    pointer-events: none !important;
}
</style>
"""

panel_html = f"""
<div id="tile-panel">
  <h3>CONUS Tile Index &mdash; {n_tiles} tiles</h3>

  <div class="section">
    <b>Selected (<span id="sel-count">0</span>)</b>
    <div id="sel-list"></div>
    <button onclick="copySelection()">Copy Tile IDs</button>
    <button class="secondary" onclick="clearSelection()">Clear Selection</button>
  </div>

  <div class="section">
    <label>Bounding box &mdash; drag on map</label>
    <button id="draw-btn" onclick="toggleDrawMode()">Draw Box to Select</button>
    <div class="status" id="draw-status"></div>
  </div>

  <div class="section">
    <label>Bounding box &mdash; type coordinates</label>
    <div class="coord-row">
      <input type="number" step="any" id="bbox-minlat" placeholder="min lat">
      <input type="number" step="any" id="bbox-minlon" placeholder="min lon">
    </div>
    <div class="coord-row">
      <input type="number" step="any" id="bbox-maxlat" placeholder="max lat">
      <input type="number" step="any" id="bbox-maxlon" placeholder="max lon">
    </div>
    <button onclick="selectByTypedBbox()">Select Tiles in Box</button>
  </div>

  <div class="section">
    <label>Coordinate lookup</label>
    <div class="coord-row">
      <input type="number" step="any" id="pt-lat" placeholder="lat">
      <input type="number" step="any" id="pt-lon" placeholder="lon">
    </div>
    <button onclick="selectByPoint()">Select Tile at Point</button>
  </div>

  <div class="section">
    <label>Find tile by ID</label>
    <input type="text" id="find-id" placeholder="e.g. h10v05">
    <button onclick="findTileById()">Find &amp; Select</button>
    <div class="meta">Click any tile to toggle it on/off directly.</div>
  </div>
</div>
"""

m.get_root().html.add_child(Element(panel_css))
m.get_root().html.add_child(Element(panel_html))

# ----------------------------------------------------------------------
# MAIN JS — selection state + bbox/point/id logic
# ----------------------------------------------------------------------
selection_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {

    // These two variables are declared by folium's own map-init script,
    // which is placed at the very end of the document. Because this whole
    // block only runs on DOMContentLoaded (i.e. after that script has
    // already executed), it is safe to reference them directly here.
    var mapObj = {{MAP_NAME}};
    var geoLayer = {{GEOJSON_LAYER_NAME}};
    var TILE_ID_FIELD = "{{TILE_ID_FIELD}}";

    // One-time diagnostics: confirms Leaflet.Draw actually loaded, since
    // a failed CDN load would otherwise fail silently later.
    if (typeof L === "undefined") console.error("Leaflet is not loaded.");
    if (typeof L.Draw === "undefined") console.error("Leaflet.Draw is NOT loaded — check network/CDN access to jsdelivr.net.");
    if (typeof L.Draw !== "undefined" && typeof L.Draw.Rectangle === "undefined") console.error("L.Draw.Rectangle is not available.");
    if (typeof turf === "undefined") console.error("turf.js is not loaded.");

    var selectedTiles = new Set();
    var layerById = {};   // tile_id -> leaflet layer
    var selectedColor = "#e74c3c";
    var defaultColor = "#3186cc";
    var hoverWeight = 1.6;
    var baseWeight = 0.6;

    geoLayer.eachLayer(function(layer) {
        var tid = layer.feature.properties[TILE_ID_FIELD];
        layerById[tid] = layer;

        layer.on('click', function() {
            toggleTile(tid);
        });

        // Hover cue that never fights the selection color: it only ever
        // touches border weight, and only while the tile is unselected.
        layer.on('mouseover', function() {
            if (!selectedTiles.has(tid)) layer.setStyle({weight: hoverWeight});
        });
        layer.on('mouseout', function() {
            if (!selectedTiles.has(tid)) layer.setStyle({weight: baseWeight});
        });
    });

    function setLayerStyle(tid, on) {
        var layer = layerById[tid];
        if (!layer) return;
        layer.setStyle(on
            ? {fillColor: selectedColor, weight: 2, fillOpacity: 0.55}
            : {fillColor: defaultColor, weight: baseWeight, fillOpacity: 0.22});
    }

    function toggleTile(tid) {
        if (selectedTiles.has(tid)) {
            selectedTiles.delete(tid);
            setLayerStyle(tid, false);
        } else {
            selectedTiles.add(tid);
            setLayerStyle(tid, true);
        }
        updatePanel();
    }

    function selectTile(tid, on) {
        if (on) selectedTiles.add(tid); else selectedTiles.delete(tid);
        setLayerStyle(tid, on);
    }

    function updatePanel() {
        document.getElementById("sel-count").innerText = selectedTiles.size;
        document.getElementById("sel-list").innerText =
            selectedTiles.size ? Array.from(selectedTiles).sort().join(", ") : "(none)";
    }

    window.clearSelection = function() {
        selectedTiles.forEach(function(tid) { setLayerStyle(tid, false); });
        selectedTiles.clear();
        updatePanel();
    };

    window.copySelection = function() {
        var text = Array.from(selectedTiles).sort().join(", ");
        navigator.clipboard.writeText(text).then(function() {
            alert("Copied " + selectedTiles.size + " tile ID(s) to clipboard");
        }, function() {
            prompt("Copy manually:", text);
        });
    };

    // ---------- bbox intersection (shared by drawn + typed boxes) ----------
    function selectTilesInBbox(minLat, minLon, maxLat, maxLon) {
        var bboxPoly = turf.bboxPolygon([minLon, minLat, maxLon, maxLat]);
        var matched = 0;
        geoLayer.eachLayer(function(layer) {
            var tid = layer.feature.properties[TILE_ID_FIELD];
            var gj = layer.toGeoJSON();
            try {
                if (turf.booleanIntersects(gj, bboxPoly)) {
                    selectTile(tid, true);
                    matched++;
                }
            } catch (e) { /* skip malformed geometry */ }
        });
        updatePanel();
        return matched;
    }

    window.selectByTypedBbox = function() {
        var minLat = parseFloat(document.getElementById("bbox-minlat").value);
        var minLon = parseFloat(document.getElementById("bbox-minlon").value);
        var maxLat = parseFloat(document.getElementById("bbox-maxlat").value);
        var maxLon = parseFloat(document.getElementById("bbox-maxlon").value);
        if ([minLat, minLon, maxLat, maxLon].some(isNaN)) {
            alert("Enter all four bounding-box values.");
            return;
        }
        var n = selectTilesInBbox(
            Math.min(minLat, maxLat), Math.min(minLon, maxLon),
            Math.max(minLat, maxLat), Math.max(minLon, maxLon)
        );
        document.getElementById("draw-status").innerText = n + " tile(s) selected.";
    };

    // ---------- point-in-polygon lookup ----------
    window.selectByPoint = function() {
        var lat = parseFloat(document.getElementById("pt-lat").value);
        var lon = parseFloat(document.getElementById("pt-lon").value);
        if (isNaN(lat) || isNaN(lon)) {
            alert("Enter both latitude and longitude.");
            return;
        }
        var pt = turf.point([lon, lat]);
        var found = null;
        geoLayer.eachLayer(function(layer) {
            if (found) return;
            var gj = layer.toGeoJSON();
            try {
                if (turf.booleanPointInPolygon(pt, gj)) {
                    found = layer.feature.properties[TILE_ID_FIELD];
                }
            } catch (e) {}
        });
        if (found) {
            selectTile(found, true);
            updatePanel();
            mapObj.flyTo([lat, lon], 8);
        } else {
            alert("No tile contains that point.");
        }
    };

    // ---------- find by tile ID ----------
    window.findTileById = function() {
        var tid = document.getElementById("find-id").value.trim();
        var layer = layerById[tid];
        if (!layer) {
            alert("Tile ID not found: " + tid);
            return;
        }
        selectTile(tid, true);
        updatePanel();
        mapObj.fitBounds(layer.getBounds(), {maxZoom: 9});
    };

    // ---------- drag-a-box (Leaflet.Draw) ----------
    var drawControl = new L.Draw.Rectangle(mapObj, {
        shapeOptions: {color: '#e74c3c', weight: 2}
    });
    var drawing = false;

    window.toggleDrawMode = function() {
        var btn = document.getElementById("draw-btn");
        if (!drawing) {
            drawControl.enable();
            drawing = true;
            btn.classList.add("active");
            btn.innerText = "Cancel Drawing";
            document.getElementById("draw-status").innerText = "Drag a rectangle on the map...";
            mapObj.getContainer().classList.add("drawing-active");
            // Diagnostics: confirm the class landed and see what's actually
            // controlling the cursor at this point.
            console.log("drawing-active added:",
                mapObj.getContainer().classList.contains("drawing-active"));
            console.log("container computed cursor:",
                getComputedStyle(mapObj.getContainer()).cursor);
        } else {
            drawControl.disable();
            drawing = false;
            btn.classList.remove("active");
            btn.innerText = "Draw Box to Select";
            mapObj.getContainer().classList.remove("drawing-active");
        }
    };

    mapObj.on(L.Draw.Event.CREATED, function(e) {
        // The drawn rectangle is only a query shape — used once to find
        // intersecting tiles, then discarded. We deliberately never add
        // it to the map: a lingering rectangle would sit on top of the
        // tiles underneath and block clicking them directly.
        var b = e.layer.getBounds();
        var n = selectTilesInBbox(
            b.getSouth(), b.getWest(), b.getNorth(), b.getEast()
        );
        document.getElementById("draw-status").innerText = n + " tile(s) selected.";
        var btn = document.getElementById("draw-btn");
        drawing = false;
        btn.classList.remove("active");
        btn.innerText = "Draw Box to Select";
        mapObj.getContainer().classList.remove("drawing-active");
    });

    updatePanel();
});
</script>
"""

selection_js = selection_js.replace("{{MAP_NAME}}", m.get_name())
selection_js = selection_js.replace("{{GEOJSON_LAYER_NAME}}", geojson_layer.get_name())
selection_js = selection_js.replace("{{TILE_ID_FIELD}}", TILE_ID_FIELD)

m.get_root().html.add_child(Element(selection_js))

# ----------------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------------
m.save(OUT_HTML)
print(f"Saved {n_tiles}-tile professional map to: {OUT_HTML}")

C:\Users\mgvhy\AppData\Roaming\Python\Python313\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'conus_tile_index.gpkg': 'conus_tile_index_clean' (default), 'conus_tile_index'. Specify layer parameter to avoid this warning.
  result = read_func(


Saved 422-tile professional map to: C:/Users/mgvhy/OneDrive - University of Missouri/scientific_data/code_index/conus_tile_map.html
